# 水果新鮮度 CNN：現場 Demo
直接載入已訓練好的模型，**不需要 GPU、也不用下載資料集**，全部跑完約 1 分鐘。

完整重新訓練請用 [colab_run.ipynb](https://colab.research.google.com/github/jerry910127/-/blob/main/colab_run.ipynb)。

## 0. 取得程式與訓練好的模型

In [ ]:
%cd /content
!rm -rf fruit_cnn && git clone -q https://github.com/jerry910127/-.git fruit_cnn
%cd /content/fruit_cnn
import json, os, sys
from IPython.display import Image, display
sys.path.insert(0, 'fruit')

## 1. 訓練結果（Colab T4，按原始照片分組切分）

In [ ]:
m = json.load(open('fruit/result/metrics.json'))
md = m['metadata']
print(f"原始照片 {md['source_photos']:,} 張 × 9 個版本｜測試洩漏：原始切分 {md['leak_original_split']:.0%} → 本次 {md['leak_this_split']:.0%}")
print(f"Test accuracy {m['test_accuracy']:.2%}｜Macro F1 {m['macro_f1']:.4f}｜參數量 {m['params']:,}｜{m['epochs_run']} 回合")
o = json.load(open('fruit/result_original_split/metrics.json'))
print(f"對照：原始切分（有洩漏）accuracy {o['test_accuracy']:.2%}")
display(Image('fruit/result/confusion_matrix.png', width=520))
display(Image('fruit/result/learning_curves.png', width=760))

## 2. 預測 24 張資料集以外的照片
檔名開頭是真實類別，可以直接對答案；照片來源見 `fruit/demo_photos/CREDITS.txt`。

In [ ]:
!python fruit/predict.py fruit/demo_photos --out /content/demo.png > /dev/null
d = json.load(open('/content/demo.json', encoding='utf-8'))
pairs = [(r['file'].rsplit('_', 1)[0], r['top3'][0][0]) for r in d]
exact = sum(t == p for t, p in pairs)
fresh = sum(t.startswith('fresh') == p.startswith('fresh') for t, p in pairs)
print(f'{len(pairs)} 張：完全正確 {exact}（{exact/len(pairs):.0%}），新鮮度正確 {fresh}（{fresh/len(pairs):.0%}）')
for (t, p), r in zip(pairs, d):
    print(f"{'✓' if t == p else '✗'} {r['file']:22s} → {p} ({r['top3'][0][1]:.0%})")
display(Image('/content/demo.png', width=900))

## 3. 上傳現場拍的照片
可一次選多張。

In [ ]:
from google.colab import files
os.makedirs('/content/live', exist_ok=True)
for name, data in files.upload().items():
    open(f'/content/live/{name}', 'wb').write(data)
!python fruit/predict.py /content/live --out /content/live.png
display(Image('/content/live.png', width=700))

## 4. 程式碼重點

In [ ]:
import inspect, dataset, cnn, train
print('=== dataset.py：去掉增強前綴，得到原始照片 id ===')
print('AUG_PREFIX =', dataset.AUG_PREFIX.pattern)
print(inspect.getsource(dataset.source_id))
print('=== cnn.py：模型架構 ===')
print(inspect.getsource(cnn.CNN))
print('=== train.py：一個 epoch 的訓練 / 驗證 ===')
print(inspect.getsource(train.run))

In [ ]:
# 分組切分的核心（dataset.prepare_data 中 split == 'group' 的部分）
src = inspect.getsource(dataset.prepare_data)
start = src.index("elif split == 'group'")
print(src[start:src.index("    else:", start)])